# The Businessman: Cooltra Use Case Investigation
*"The numbers never lie. But they do whisper secrets to those clever enough to listen."*

## Case File
- **Use Case**: Cooltra-Bot (Cortex Analyst)
- **Account**: Felyx
- **Go-Live Date**: 2025-12-17
- **Stated EACV**: $5,000
- **Prioritized Features**: AI - Cortex Analyst, AI - Basic

**Objective**: Find the gap between stated ACV and true consumption value by tracing upstream costs.

In [ ]:
import os
import snowflake.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse")
cursor = conn.cursor()

GO_LIVE_DATE = '2025-12-17'
BASELINE_START = '2025-09-17'
ANALYSIS_END = '2026-02-10'
USE_CASE_NAME = 'Cooltra-Bot'
STATED_EACV = 5000

print(f"Investigation opened: {USE_CASE_NAME}")
print(f"Go-Live: {GO_LIVE_DATE}")
print(f"Stated EACV: ${STATED_EACV:,}")

## Phase 1: Confirm the Target
*"Rule 194: It's always good business to know about new customers before they walk in your door."*

In [ ]:
use_case_query = """
SELECT 
    USE_CASE_ID,
    USE_CASE_NAME,
    ACCOUNT_NAME,
    GO_LIVE_DATE,
    USE_CASE_EACV,
    PRIORITIZED_FEATURES,
    USE_CASE_STAGE,
    REGION_NAME
FROM MDM.MDM_INTERFACES.DIM_USE_CASE
WHERE USE_CASE_NAME ILIKE '%Cooltra%'
  AND USE_CASE_STAGE = '7 - Deployed'
"""
cursor.execute(use_case_query)
use_case_df = cursor.fetch_pandas_all()
print("Case File Confirmed:")
use_case_df

## Phase 2: Establish the Baseline
*"Rule 22: A wise man can hear profit in the wind. But first, you need to know what silence sounds like."*

We look at consumption 90 days before go-live to establish "normal" patterns.

In [ ]:
consumption_query = f"""
WITH daily_consumption AS (
    SELECT 
        usage_date,
        service_type,
        SUM(credits_used) as credits_used,
        SUM(credits_billed) as credits_billed
    FROM snowflake.account_usage.metering_daily_history
    WHERE usage_date BETWEEN '{BASELINE_START}' AND '{ANALYSIS_END}'
    GROUP BY 1, 2
)
SELECT 
    usage_date,
    service_type,
    credits_used,
    credits_billed,
    CASE 
        WHEN usage_date < '{GO_LIVE_DATE}' THEN 'PRE-GOLIVE'
        ELSE 'POST-GOLIVE'
    END as period
FROM daily_consumption
ORDER BY usage_date, service_type
"""
cursor.execute(consumption_query)
consumption_df = cursor.fetch_pandas_all()
print(f"Retrieved {len(consumption_df)} consumption records")
consumption_df.head(10)

In [ ]:
ai_services = ['AI_SERVICES', 'CORTEX_ANALYST', 'SNOWPARK_CONTAINER_SERVICES']
ai_consumption = consumption_df[consumption_df['SERVICE_TYPE'].isin(ai_services)].copy()

fig = px.area(
    ai_consumption,
    x='USAGE_DATE',
    y='CREDITS_USED',
    color='SERVICE_TYPE',
    title='Direct AI/ML Consumption Over Time',
    labels={'CREDITS_USED': 'Credits Used', 'USAGE_DATE': 'Date'}
)
fig.add_vline(x=GO_LIVE_DATE, line_dash="dash", line_color="red", annotation_text="Go-Live")
fig.update_layout(height=500)
fig.show()

## Phase 3: Follow the Latinum
*"Here's where it gets interesting. The amateurs look at the obvious. I look at the connections."*

In [ ]:
warehouse_query = f"""
SELECT 
    DATE_TRUNC('day', start_time) as usage_date,
    warehouse_name,
    SUM(credits_used) as credits_used
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time BETWEEN '{BASELINE_START}' AND '{ANALYSIS_END}'
GROUP BY 1, 2
ORDER BY 1, 3 DESC
"""
cursor.execute(warehouse_query)
warehouse_df = cursor.fetch_pandas_all()
print(f"Retrieved {len(warehouse_df)} warehouse consumption records")

warehouse_daily = warehouse_df.groupby('USAGE_DATE')['CREDITS_USED'].sum().reset_index()
warehouse_daily['PERIOD'] = warehouse_daily['USAGE_DATE'].apply(
    lambda x: 'PRE-GOLIVE' if x < pd.Timestamp(GO_LIVE_DATE) else 'POST-GOLIVE'
)

fig = px.bar(
    warehouse_daily,
    x='USAGE_DATE',
    y='CREDITS_USED',
    color='PERIOD',
    title='Upstream: Warehouse Consumption (Potential AI/ML Driver)',
    color_discrete_map={'PRE-GOLIVE': '#636EFA', 'POST-GOLIVE': '#00CC96'}
)
fig.add_vline(x=GO_LIVE_DATE, line_dash="dash", line_color="red", annotation_text="Go-Live")
fig.update_layout(height=500)
fig.show()

In [ ]:
pre_post_query = f"""
WITH service_summary AS (
    SELECT 
        service_type,
        CASE WHEN usage_date < '{GO_LIVE_DATE}' THEN 'PRE' ELSE 'POST' END as period,
        SUM(credits_used) as total_credits,
        AVG(credits_used) as daily_avg,
        COUNT(DISTINCT usage_date) as active_days
    FROM snowflake.account_usage.metering_daily_history
    WHERE usage_date BETWEEN '{BASELINE_START}' AND '{ANALYSIS_END}'
    GROUP BY 1, 2
)
SELECT 
    service_type,
    MAX(CASE WHEN period = 'PRE' THEN total_credits END) as pre_total,
    MAX(CASE WHEN period = 'POST' THEN total_credits END) as post_total,
    MAX(CASE WHEN period = 'PRE' THEN daily_avg END) as pre_daily_avg,
    MAX(CASE WHEN period = 'POST' THEN daily_avg END) as post_daily_avg,
    COALESCE(MAX(CASE WHEN period = 'POST' THEN total_credits END), 0) - 
        COALESCE(MAX(CASE WHEN period = 'PRE' THEN total_credits END), 0) as credit_delta
FROM service_summary
GROUP BY 1
ORDER BY credit_delta DESC
"""
cursor.execute(pre_post_query)
pre_post_df = cursor.fetch_pandas_all()
print("Pre vs Post Go-Live Comparison:")
pre_post_df

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Pre Go-Live Daily Avg', 'Post Go-Live Daily Avg'),
    specs=[[{'type': 'pie'}, {'type': 'pie'}]]
)

pre_data = pre_post_df[pre_post_df['PRE_DAILY_AVG'].notna()]
post_data = pre_post_df[pre_post_df['POST_DAILY_AVG'].notna()]

fig.add_trace(
    go.Pie(labels=pre_data['SERVICE_TYPE'], values=pre_data['PRE_DAILY_AVG'], name='Pre'),
    row=1, col=1
)
fig.add_trace(
    go.Pie(labels=post_data['SERVICE_TYPE'], values=post_data['POST_DAILY_AVG'], name='Post'),
    row=1, col=2
)

fig.update_layout(title_text='Consumption Mix: Before vs After Go-Live', height=500)
fig.show()

## Phase 4: Build the Correlation Matrix
*"See? The lobes are developing nicely. Now let's see what's REALLY connected."*

In [ ]:
pivot_consumption = consumption_df.pivot_table(
    index='USAGE_DATE',
    columns='SERVICE_TYPE',
    values='CREDITS_USED',
    aggfunc='sum'
).fillna(0)

warehouse_totals = warehouse_df.groupby('USAGE_DATE')['CREDITS_USED'].sum()
warehouse_totals.name = 'WAREHOUSE_TOTAL'

analysis_df = pivot_consumption.join(warehouse_totals, how='outer').fillna(0)

post_golive_df = analysis_df[analysis_df.index >= pd.Timestamp(GO_LIVE_DATE)]

correlation_matrix = post_golive_df.corr()

fig = px.imshow(
    correlation_matrix,
    title='Consumption Correlation Matrix (Post Go-Live)',
    color_continuous_scale='RdBu',
    aspect='auto'
)
fig.update_layout(height=600)
fig.show()

## Phase 5: Calculate True Value
*"This is the moment, partner. The moment we find out what this use case is REALLY worth."*

In [ ]:
ai_service_types = ['AI_SERVICES', 'CORTEX_ANALYST']

ai_direct = pre_post_df[pre_post_df['SERVICE_TYPE'].isin(ai_service_types)]['CREDIT_DELTA'].sum()
ai_direct = max(ai_direct, 0)

warehouse_pre = warehouse_daily[warehouse_daily['PERIOD'] == 'PRE-GOLIVE']['CREDITS_USED'].mean() or 0
warehouse_post = warehouse_daily[warehouse_daily['PERIOD'] == 'POST-GOLIVE']['CREDITS_USED'].mean() or 0
warehouse_delta = warehouse_post - warehouse_pre

if 'AI_SERVICES' in correlation_matrix.columns and 'WAREHOUSE_TOTAL' in correlation_matrix.columns:
    attribution_factor = max(correlation_matrix.loc['AI_SERVICES', 'WAREHOUSE_TOTAL'], 0)
else:
    attribution_factor = 0.3

attributed_upstream = warehouse_delta * attribution_factor * 55 if warehouse_delta > 0 else 0

total_credits = ai_direct + attributed_upstream

credit_rate = 3.00
calculated_value = total_credits * credit_rate * 12

print("="*60)
print("THE BUSINESSMAN'S VALUATION REPORT")
print("="*60)
print(f"\nDIRECT AI/ML CONSUMPTION")
print(f"   New AI Services Credits (delta): {ai_direct:,.2f}")
print(f"\nCORRELATED UPSTREAM CONSUMPTION")
print(f"   Warehouse Daily Avg Delta: {warehouse_delta:,.2f} credits")
print(f"   Attribution Factor: {attribution_factor:.2%}")
print(f"   Attributed Credits (annualized): {attributed_upstream:,.2f}")
print(f"\nVALUATION")
print(f"   Total Credits Attributable: {total_credits:,.2f}")
print(f"   Annualized Value (@ ${credit_rate}/credit): ${calculated_value:,.2f}")
print(f"\nGAP ANALYSIS")
print(f"   Stated EACV: ${STATED_EACV:,}")
print(f"   Calculated Value: ${calculated_value:,.2f}")
gap = calculated_value - STATED_EACV
print(f"   Gap: ${gap:,.2f}")
if STATED_EACV > 0:
    print(f"   Gap %: {(gap / STATED_EACV * 100):.1f}%")
print("="*60)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Calculated Value',
    x=['Use Case Value'],
    y=[calculated_value],
    marker_color='#00CC96',
    text=[f'${calculated_value:,.0f}'],
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Stated EACV',
    x=['Use Case Value'],
    y=[STATED_EACV],
    marker_color='#636EFA',
    text=[f'${STATED_EACV:,}'],
    textposition='inside'
))

fig.update_layout(
    title='The Verdict: Stated EACV vs True Value',
    barmode='group',
    height=400,
    yaxis_title='Annual Value ($)'
)
fig.show()

In [ ]:
breakdown_fig = go.Figure(go.Waterfall(
    name="Value Breakdown",
    orientation="v",
    measure=["relative", "relative", "total", "relative"],
    x=["Direct AI/ML", "Attributed Upstream", "Calculated Total", "vs Stated EACV"],
    y=[ai_direct * credit_rate * 12, 
       attributed_upstream * credit_rate * 12, 
       0,
       STATED_EACV - calculated_value],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    text=[f'${ai_direct * credit_rate * 12:,.0f}',
          f'${attributed_upstream * credit_rate * 12:,.0f}',
          f'${calculated_value:,.0f}',
          f'${STATED_EACV - calculated_value:,.0f}']
))

breakdown_fig.update_layout(
    title="Value Waterfall: How We Got to True Value",
    showlegend=False,
    height=500
)
breakdown_fig.show()

## The Verdict
*"And THERE it is. The latinum they forgot to count."*

### Rule 162: "Even in the worst of times, someone turns a profit."

This notebook provides the forensic trail. The numbers don't lie - they just needed someone with the lobes to listen.

In [ ]:
conn.close()
print("\nBooks closed. Investigation complete.")
print("\n*adjusts jacket* \"You know, with a little work on those ears, you could pass for one of us.\"")